# Example 5 — Deep Inelastic Scattering (dipole portal)

**Status: work in progress (Milestone 1 — cross sections).**

This notebook documents the deep-inelastic scattering (DIS) regime for the **dipole / transition-magnetic-moment portal**, following

> Huang, Jana, Lindner, Rodejohann, *Probing Heavy Sterile Neutrinos at Ultrahigh Energy Neutrino Telescopes via the Dipole Portal* (arXiv:2204.xxxx).

At ultrahigh neutrino energies the exchanged photon can resolve the quarks inside the nucleon, and $\nu \to N$ conversion proceeds via DIS in addition to the coherent and diffractive (quasi-elastic) regimes already in DarkNews.

## Scope
- **Milestone 1 (this notebook): the DIS cross section** — $\mathrm{d}^2\sigma/\mathrm{d}x\,\mathrm{d}y$, $\mathrm{d}\sigma/\mathrm{d}y$, and $\sigma(E_\nu)$. This is what the paper computes (for attenuation / rate estimates).
- **Milestone 2 (future): exclusive DIS event generation** — final-state parton/jet kinematics, hadronic recoil — is a larger follow-up and is *not* implemented yet.

## Design (kept close to the existing DarkNews structure)
| Piece | Location |
|---|---|
| DIS kinematics `shat`, `Q2`, and $x,y$ limits | `DarkNews.phase_space` |
| PDF interface (pluggable) + EM structure function | `DarkNews.pdf` |
| Partonic target derived from the nucleus | `NuclearTarget.get_constituent_quarks()` |
| Dipole DIS $\mathrm{d}^2\sigma/\mathrm{d}x\,\mathrm{d}y$ | `DarkNews.amplitudes.dis_diff_xsec_dxdy` |

The target is still the nucleus (`get_constituent_quarks()` returns a partonic view that keeps `Z, N, A`), and the scattering regime is selected with `scattering_regime="DIS"`, exactly like `"coherent"`, `"p-el"`, `"n-el"`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from DarkNews import phase_space as ps
from DarkNews import pdf as dnpdf
from DarkNews import amplitudes as amps
from DarkNews import pdg
from DarkNews.nuclear_tools import NuclearTarget
from DarkNews.model import ThreePortalModel
from DarkNews.processes import UpscatteringProcess

_trapz = getattr(np, 'trapezoid', getattr(np, 'trapz', None))  # np.trapz removed in numpy 2

## 1) DIS kinematics and limits

For a parton carrying momentum fraction $x$ of a nucleon of mass $M$:
$$ \hat s = 2 M E_\nu x + M^2 x^2, \qquad Q^2 = 2 M E_\nu x\, y, $$
with $x$ the Bjorken variable and $y$ the inelasticity. The allowed ranges follow the paper's appendix (both exact and approximate forms are implemented). The DIS regime is defined by the hard cut $Q^2 > 2~\mathrm{GeV}^2$.

In [ ]:
Enu = 1e9        # 1 EeV
M   = 0.938      # nucleon mass [GeV]
mHNL = 100.0     # HNL mass [GeV]

print('x_min  exact :', ps.dis_xmin(Enu, mHNL, M))
print('x_min  approx:', ps.dis_xmin(Enu, mHNL, M, exact=False))

x = np.geomspace(ps.dis_xmin(Enu, mHNL, M), 1.0, 6)
ymin, ymax = ps.dis_ylimits(Enu, x, mHNL, M)
for xi, lo, hi in zip(x, ymin, ymax):
    print(f'x={xi:.2e}  y in [{max(lo,0):.3e}, {hi:.3f}]')

## 2) PDF interface

DarkNews does **not** bundle a PDF set. `DarkNews.pdf` defines a small backend-agnostic interface:
- **`dnpdf.mkPDF("CT18NNLO")`** — the recommended pure-Python backend (`PartonPDFSet`, via the [`parton`](https://github.com/DavidMStraub/parton) package; reads standard LHAPDF grids, no C++/LHAPDF);
- `LHAPDFSet("CT18NNLO")` — wraps LHAPDF if you already have it;
- `CallablePDF(func)` — wraps any `func(pid, x, Q2) -> x f(x, Q2)`;
- `UnavailablePDF()` — the default placeholder; **raises** if evaluated so a DIS run can never silently use a missing PDF.

Install the backend and a set once with:
```
pip install "DarkNews[dis]"
python -m parton install CT18NNLO
```

The EM structure function $\sum_i e_i^2\,q_i(x,Q^2)$, summed over the $Z$ protons and $N$ neutrons of the nucleus (neutron by isospin), is `pdf.em_structure_function`.

In [ ]:
PDF_SET = 'CT18NNLO'

# Recommended: a real set via the pure-Python `parton` backend. Fall back to a
# crude toy PDF (NOT physics) if the set isn't installed, so the notebook still runs.
try:
    pdf = dnpdf.mkPDF(PDF_SET)
    USING_REAL_PDF = True
    print(f'Using real PDF set: {PDF_SET}')
except Exception as exc:
    print(f"Could not load '{PDF_SET}' ({type(exc).__name__}); falling back to a TOY PDF (not physics).")
    print("  -> install with:  pip install 'DarkNews[dis]'  &&  python -m parton install", PDF_SET)

    def _toy_xf(pid, x, Q2):
        x = np.asarray(x, float)
        return np.where(abs(pid) in (1, 2), 0.3 * (1 - x) ** 3, 0.05 * (1 - x) ** 5)

    pdf = dnpdf.CallablePDF(_toy_xf)
    USING_REAL_PDF = False

F = dnpdf.em_structure_function(pdf, x, ps.dis_Q2(Enu, x, 0.5, M), Z=8, N=8)  # 16O
print('structure function on 16O:', F)

In [ ]:
# Visualize the parton content: x f(x, Q2) vs x for a few flavors, at Q = 10 GeV.
xgrid = np.geomspace(1e-4, 0.9, 200)
Q2plot = np.full_like(xgrid, 10.0 ** 2)
flavors = {'u': 2, 'd': 1, r'$\bar u$': -2, r'$\bar d$': -1, 's': 3, 'c': 4}

fig, ax = plt.subplots(figsize=(7, 4))
for label, pid in flavors.items():
    ax.plot(xgrid, pdf.xfxQ2(pid, xgrid, Q2plot), label=label)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('x'); ax.set_ylabel(r'$x\,f(x, Q^2)$')
ax.set_ylim(1e-3, 5)
ax.set_title(f"{'CT18NNLO' if USING_REAL_PDF else 'TOY PDF'} — Q = 10 GeV")
ax.legend(ncol=3, fontsize=9)
plt.tight_layout(); plt.show()

## 3) Partonic target and the DIS process

`get_constituent_quarks(pdf=...)` returns a partonic view of the nucleus (keeps `Z, N, A`, uses the nucleon mass scale). Selecting `scattering_regime="DIS"` on `UpscatteringProcess` uses it automatically.

In [ ]:
O16 = NuclearTarget('O16')
model = ThreePortalModel(m4=mHNL, mu_tr_mu4=1e-6, mzprime=1.25)

proc = UpscatteringProcess(
    nu_projectile=pdg.numu, nu_upscattered=pdg.neutrino4,
    nuclear_target=O16, scattering_regime='DIS',
    TheoryModel=model, helicity='conserving',
)
proc.target.pdf = pdf   # attach the PDF (until GenLauncher wiring lands, see below)
print('target:', proc.target.name, '| Z,N,A =', proc.target.Z, proc.target.N, proc.target.A)

## 4) Dipole DIS differential cross section

$$ \frac{\mathrm{d}^2\sigma}{\mathrm{d}x\,\mathrm{d}y} = 2 M E_\nu x \;\frac{\mathrm{d}\sigma_{\rm parton}(x)}{\mathrm{d}t}\; \sum_i e_i^2 q_i(x, Q^2), $$
with the parton-level dipole $\mathrm{d}\sigma/\mathrm{d}t$ (elastic result at $F_1=1, F_2=0$, $M\to Mx$, $s\to\hat s$). The dipole coupling uses DarkNews' `Tij` ($=\mu_{\rm tr}/2$), consistent with the elastic `TMM_SQR` diagram.

Below we plot $y\,\mathrm{d}\sigma/\mathrm{d}y$ (as in the paper's Fig. 2) at fixed $E_\nu$, integrating over $x$ at each $y$.

In [ ]:
def dsigma_dy(Enu, y, proc, nx=200):
    """dsigma/dy at fixed y, integrating over x. Applies the DIS cut Q2 > 2 GeV^2."""
    M = proc.target.mass
    xlo = ps.dis_xmin(Enu, proc.m_ups, M)
    xs = np.geomspace(max(xlo, 1e-6), 1.0, nx)
    ymin, ymax = ps.dis_ylimits(Enu, xs, proc.m_ups, M)
    integrand = amps.dis_diff_xsec_dxdy(Enu, xs, np.full_like(xs, y), proc)
    Q2 = ps.dis_Q2(Enu, xs, y, M)
    mask = (y > np.maximum(ymin, 0)) & (y < ymax) & (Q2 > 2.0)
    return _trapz(np.where(mask, integrand, 0.0), xs)


ys = np.geomspace(1e-6, 1.0, 60)
dsdy = np.array([dsigma_dy(Enu, y, proc) for y in ys])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ys, ys * dsdy)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('y'); ax.set_ylabel(r'$y\,\mathrm{d}\sigma/\mathrm{d}y$  [cm$^2$]')
_tag = 'CT18NNLO' if USING_REAL_PDF else 'toy PDF'
ax.set_title(rf'Dipole DIS ({_tag}) — $E_\nu=1$ EeV, $m_N=100$ GeV, $^{{16}}$O')
plt.tight_layout(); plt.show()

## 5) Total cross section $\sigma(E_\nu)$

Integrating $\mathrm{d}^2\sigma/\mathrm{d}x\,\mathrm{d}y$ over the allowed $(x,y)$ (with the $Q^2>2~\mathrm{GeV}^2$ cut) gives the total DIS cross section vs neutrino energy — the analogue of the paper's Fig. 3. Each curve has a threshold at $E_\nu^{\rm min}\approx m_N^2/(2M)$ and then rises with energy as the sea-quark density grows. (Grid integral here for illustration; the generator will use vegas.)

In [ ]:
def total_xsec(Enu, proc, nx=60, ny=60):
    """Total DIS cross section [cm^2]: integrate d2sigma/dx dy over the allowed
    (x, y) with the Q2 > 2 GeV^2 cut."""
    M, mHNL = proc.target.mass, proc.m_ups
    xlo = ps.dis_xmin(Enu, mHNL, M)
    if xlo >= 1.0:
        return 0.0
    xs = np.geomspace(max(xlo, 1e-6), 1.0, nx)
    dsdx = np.zeros_like(xs)
    for i, xv in enumerate(xs):
        ymin, ymax = ps.dis_ylimits(Enu, xv, mHNL, M)
        ylo = max(max(ymin, 0.0), 2.0 / (2 * M * Enu * xv))  # Q2 > 2 GeV^2 cut
        if ylo >= ymax:
            continue
        ys = np.geomspace(ylo, ymax, ny)
        d2 = amps.dis_diff_xsec_dxdy(Enu, np.full_like(ys, xv), ys, proc)
        dsdx[i] = _trapz(d2, ys)
    return _trapz(dsdx, xs)


def make_dis_process(mN):
    mdl = ThreePortalModel(m4=mN, mu_tr_mu4=1e-6, mzprime=1.25)
    p = UpscatteringProcess(nu_projectile=pdg.numu, nu_upscattered=pdg.neutrino4,
                            nuclear_target=O16, scattering_regime='DIS',
                            TheoryModel=mdl, helicity='conserving')
    p.target.pdf = pdf
    return p


Enus = np.geomspace(1e5, 1e10, 12)
fig, ax = plt.subplots(figsize=(7, 4.5))
for mN in [200.0, 1000.0, 10000.0]:
    p = make_dis_process(mN)
    sig = np.array([total_xsec(E, p) for E in Enus])
    ax.plot(Enus, sig, marker='.', label=rf'$m_N = {mN/1e3:g}$ TeV')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$E_\nu$ [GeV]'); ax.set_ylabel(r'$\sigma^{\rm DIS}_{\nu\to N}$  [cm$^2$]')
_tag = 'CT18NNLO' if USING_REAL_PDF else 'toy PDF'
ax.set_title(rf'Dipole DIS total cross section ({_tag}) on $^{{16}}$O, $\mu_{{\rm tr}}=10^{{-6}}$')
ax.legend()
plt.tight_layout(); plt.show()

## Validation targets (before this is used for physics)
1. **Reproduce the paper's Fig. 2** ($y\,\mathrm{d}\sigma/\mathrm{d}y$ vs $y$ for $m_N = 0.1, 1, 10$ TeV on $^{16}$O, $E_\nu = 1$ EeV) and **Fig. 3** ($\sigma$ vs $E_\nu$) quantitatively — the plots above have the right qualitative shape but the **absolute dipole normalization is not yet validated**.
2. **Cross-check the dipole normalization** against the elastic `TMM_SQR` diagram in an overlapping regime, and confirm the $\mu_{\rm tr}$ ↔ paper-$\mu_{\nu N}$ mapping.
3. Consider **running $\alpha_{\rm QED}(Q^2)$** and the treatment near the $Q^2 = 2~\mathrm{GeV}^2$ DIS boundary.

## Next implementation step: vegas wiring (the extra integration variable)
The elastic regimes integrate over **one** scattering variable ($Q^2$). DIS needs **two** ($x$ and $y$). The plan:
- add a dedicated `UpscatteringXsecDIS` integrand in `DarkNews.integrands` (dim = 2: $x, y$; energy adds one more in the combined event integrand), reusing `phase_space.dis_ylimits` for the per-$x$ limits and applying the $Q^2 > 2\,\mathrm{GeV}^2$ cut;
- route `UpscatteringProcess.scalar_total_xsec` / `total_xsec` to it when `scattering_regime == "DIS"`, leaving the elastic path untouched;
- thread a `dis_pdf=` argument through `GenLauncher` so the PDF set reaches `get_constituent_quarks`.